# CityPulse v3 — train, calibrate, evaluate, export

Runs the full detector on your **real `city_data.csv`** (Open-Meteo rainfall, temperature and AQI + simulated traffic and incidents) and writes the `data/` folder the dashboard reads.

**What this fixes compared with v2.4 / v2.5**
- Isolation Forest and LSTM learn on *deviation from typical* (hour-aware z-scores + 15-minute change), not raw values.
- LSTM error uses the latest 15 minutes per feed, so an anomaly is no longer diluted across the 1-hour window.
- Both models are calibrated on **normal** validation rows only. In v2.4 the event inside validation set the ceiling, which squashed the LSTM to about 0.
- Fusion weights and all thresholds are chosen on **validation**, frozen, then applied once to **test**. v2.5 tuned thresholds on the test set.
- 7 labelled scenarios + 4 decoys, so metrics are not based on a single event.
- Every row is re-scored with each feed removed, so the dashboard's outage switch is real.

Runtime: about 3–5 minutes on the free Colab CPU.

## 1. Get the project
Either clone your GitHub repo (set the URL), or upload `citypulse_v3.zip` in the file panel and unzip it.

In [ ]:
REPO_URL = ""   # e.g. "https://github.com/<you>/citypulse.git"  (leave empty if you uploaded the zip)

import os, subprocess
if REPO_URL:
    !git clone -q {REPO_URL} citypulse
elif os.path.exists("citypulse_v3.zip"):
    !unzip -q -o citypulse_v3.zip
%cd citypulse
!ls

## 2. Upload your `city_data.csv`
(the file exported by `CityPulse_Core_Engine.ipynb`)

In [ ]:
from google.colab import files
up = files.upload()
CITY_CSV = next(iter(up))
print("Using", CITY_CSV)

## 3. Train, calibrate, evaluate, export

In [ ]:
import sys, importlib
sys.path.insert(0, "."); sys.path.insert(0, "pipeline")
import citypulse_core, citypulse_pipeline
importlib.reload(citypulse_core); importlib.reload(citypulse_pipeline)

df, meta, events = citypulse_pipeline.main(
    CITY_CSV, "data", source="Open-Meteo (real rainfall, temperature and AQI)")

## 4. Results on the unseen test period

In [ ]:
import pandas as pd, json
m = meta["metrics"]["test"]
print("Fusion weights :", meta["weights"])
print("Thresholds     :", meta["thresholds"])
print(f"Events caught  : {m['events']['detected']} / {m['events']['true_events']}")
print(f"False alarms   : {m['events']['false_alarm_episodes']}  ({m['events']['false_alarms_per_day']:.2f} per day)")
print(f"Mean minutes to confirm : {m['events']['mean_minutes_to_confirm']}")
print(f"Decoys that fired       : {m['events']['decoys_fired']}")
display(pd.DataFrame(m["events"]["per_event"]))
display(pd.DataFrame(m["ablation"]).T.rename_axis("detector"))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
test = df[df.split == "test"]
for ax, col, name in zip(axes, ["stat_score", "iso_score", "lstm_score"], ["Statistics", "Isolation Forest", "LSTM autoencoder"]):
    ax.hist(test.loc[test.ground_truth_event == 0, col], bins=40, alpha=.6, label="normal", density=True)
    ax.hist(test.loc[test.ground_truth_event == 1, col], bins=40, alpha=.6, label="event", density=True)
    ax.set_title(f"{name}: separation on test"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# City-wide combined score over the test period, with the labelled events shaded
city = test.groupby("timestamp").hybrid.max()
plt.figure(figsize=(16, 4)); plt.plot(city.index, city.values, lw=1)
for k, c in [("watch", "gold"), ("confirmed", "orange"), ("critical", "red")]:
    plt.axhline(meta["thresholds"][k], ls="--", c=c, label=k)
for p in m["events"]["per_event"]:
    plt.axvspan(pd.Timestamp(p["start"]), pd.Timestamp(p["end"]), color="red", alpha=.12)
plt.legend(); plt.title("Max combined score across zones (test)"); plt.show()

## 5. Download the `data/` folder
Replace `data/` in your GitHub repo with these files, commit, and Streamlit Cloud redeploys automatically.

In [ ]:
!zip -q -r citypulse_data.zip data -x "data/lstm_autoencoder.keras"
files.download("citypulse_data.zip")